# Part 2

In [295]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from collections import Counter
from sklearn.linear_model import LinearRegression
import numpy as np
from sklearn.metrics import mean_squared_log_error

In [296]:
train_data = pd.read_csv('my_train.csv')
x_train_data = train_data.drop(columns=["Id", "SalePrice"]).astype(str)
y_train_data = np.log(train_data['SalePrice'])

dev_data = pd.read_csv('my_dev.csv')
x_dev_data = dev_data.drop(columns=["Id", "SalePrice"]).astype(str)
y_dev_data = np.log(dev_data['SalePrice'])

In [297]:
encoder = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
encoder.fit(x_train_data)

x_train_binarized = encoder.transform(x_train_data)
x_dev_binarized = encoder.transform(x_dev_data)

feature_names = encoder.get_feature_names_out()
field_counts = Counter([name.split('_')[0] for name in feature_names])
print('Number of features: ', len(feature_names))

Number of features:  7226


In [298]:
for field, count in field_counts.items():
    print(f"Field '{field}' has {count} binary features.")

Field 'MSSubClass' has 15 binary features.
Field 'MSZoning' has 5 binary features.
Field 'LotFrontage' has 108 binary features.
Field 'LotArea' has 989 binary features.
Field 'Street' has 2 binary features.
Field 'Alley' has 3 binary features.
Field 'LotShape' has 4 binary features.
Field 'LandContour' has 4 binary features.
Field 'Utilities' has 2 binary features.
Field 'LotConfig' has 5 binary features.
Field 'LandSlope' has 3 binary features.
Field 'Neighborhood' has 25 binary features.
Field 'Condition1' has 9 binary features.
Field 'Condition2' has 8 binary features.
Field 'BldgType' has 5 binary features.
Field 'HouseStyle' has 8 binary features.
Field 'OverallQual' has 10 binary features.
Field 'OverallCond' has 9 binary features.
Field 'YearBuilt' has 110 binary features.
Field 'YearRemodAdd' has 61 binary features.
Field 'RoofStyle' has 6 binary features.
Field 'RoofMatl' has 8 binary features.
Field 'Exterior1st' has 15 binary features.
Field 'Exterior2nd' has 16 binary featu

In [299]:
model = LinearRegression()
model.fit(x_train_binarized, y_train_data)
y_dev_pred = model.predict(x_dev_binarized)
rmsle = np.sqrt(mean_squared_log_error(np.exp(y_dev_data), np.exp(y_dev_pred)))
print("RMSLE on dev:", rmsle)

RMSLE on dev: 0.15190758719398437


In [300]:
coefficients = model.coef_
coeff_df = pd.DataFrame({
    "Feature_Names": feature_names,
    "Coefficients": coefficients
})
top_positive = coeff_df.nlargest(10, "Coefficients")
top_negative = coeff_df.nsmallest(10, "Coefficients")
print("Top 10 Most Positive Features:")
print(top_positive)

print("\nTop 10 Most Negative Features:")
print(top_negative)

Top 10 Most Positive Features:
             Feature_Names  Coefficients
5901            FullBath_3      0.139358
1204         OverallQual_9      0.138181
1162  Neighborhood_StoneBr      0.125113
4816          2ndFlrSF_472      0.113359
1203         OverallQual_8      0.107227
1398      RoofMatl_WdShngl      0.092099
5184        GrLivArea_1192      0.091004
1155  Neighborhood_NoRidge      0.087031
878           LotArea_8029      0.086015
6061          GarageCars_3      0.085946

Top 10 Most Negative Features:
          Feature_Names  Coefficients
15     MSZoning_C (all)     -0.192626
5876      GrLivArea_968     -0.126873
7011  EnclosedPorch_236     -0.122768
1198      OverallQual_3     -0.114741
907        LotArea_8281     -0.108153
2444     BsmtFinSF2_311     -0.108153
1207      OverallCond_3     -0.101463
6059       GarageCars_1     -0.093753
1195      OverallQual_1     -0.089260
698        LotArea_5000     -0.087548


In [301]:
bias_weight = model.intercept_
print(f"\nBias Weight: {bias_weight}")


Bias Weight: 12.174742505694372


In [302]:
test_data = pd.read_csv("test.csv")
x_test_data = test_data.drop(columns=["Id"]).astype(str)
x_test_binarized = encoder.transform(x_test_data)
y_test_pred_log = model.predict(x_test_binarized)
y_test_pred = np.exp(y_test_pred_log)

predicted_data = test_data[["Id"]].copy()
predicted_data["SalePrice"] = y_test_pred
predicted_data = predicted_data.fillna("NA")
predicted_data.to_csv("Linear_regression_p2.csv", index=False)
print("Predicted values saved to 'Linear_regression_p2.csv'")

Predicted values saved to 'Linear_regression_p2.csv'


# Part 3

In [303]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from collections import Counter
from sklearn.linear_model import LinearRegression
import numpy as np
from sklearn.metrics import mean_squared_log_error
from sklearn.compose import ColumnTransformer

In [304]:
numerical_columns = [
    "MSSubClass", "LotArea", "OverallQual", "OverallCond", "YearBuilt", 
    "YearRemodAdd", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", 
    "TotalBsmtSF", "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "GrLivArea", 
    "BsmtFullBath", "BsmtHalfBath", "FullBath", "HalfBath", "BedroomAbvGr", 
    "KitchenAbvGr", "TotRmsAbvGrd", "Fireplaces", "GarageCars", 
    "GarageArea", "WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch", 
    "ScreenPorch", "PoolArea", "MiscVal", "MoSold", "YrSold", "LotFrontage", "GarageYrBlt", "MasVnrArea"
]

all_columns = train_data.columns.tolist()
categorical_columns = [col for col in all_columns if col not in ["Id", "SalePrice"] + numerical_columns]

x_train_data[numerical_columns] = x_train_data[numerical_columns].astype(float).fillna(0)
x_dev_data[numerical_columns] = x_dev_data[numerical_columns].astype(float).fillna(0)
x_train_data[categorical_columns] = x_train_data[categorical_columns].astype(str).fillna('NA')
x_dev_data[categorical_columns] = x_dev_data[categorical_columns].astype(str).fillna('NA')


In [305]:
num_processor = 'passthrough' 
cat_processor = OneHotEncoder(sparse_output=True, handle_unknown='ignore')

encoder = ColumnTransformer([
        ('num', num_processor, numerical_columns),
        ('cat', cat_processor, categorical_columns)
    ]
)
encoder.fit(x_train_data)

x_train_binarized = encoder.transform(x_train_data)
x_dev_binarized = encoder.transform(x_dev_data)

feature_names = encoder.get_feature_names_out()
print('Number of features: ', len(feature_names))

Number of features:  302


In [306]:
model = LinearRegression()
model.fit(x_train_binarized, y_train_data)
y_dev_pred = model.predict(x_dev_binarized)
rmsle = np.sqrt(mean_squared_log_error(np.exp(y_dev_data), np.exp(y_dev_pred)))
print("RMSLE on dev:", rmsle)

RMSLE on dev: 0.13247479018808292


In [307]:
coefficients = model.coef_
coeff_df = pd.DataFrame({
    "Feature_Names": feature_names,
    "Coefficients": coefficients
})
top_positive = coeff_df.nlargest(10, "Coefficients")
top_negative = coeff_df.nsmallest(10, "Coefficients")
print("Top 10 Most Positive Features:")
print(top_positive)
print("\nTop 10 Most Negative Features:")
print(top_negative)

Top 10 Most Positive Features:
                 Feature_Names  Coefficients
70   cat__Neighborhood_Crawfor      0.159039
86   cat__Neighborhood_StoneBr      0.145920
80   cat__Neighborhood_NridgHt      0.114277
136   cat__Exterior1st_BrkFace      0.091759
79   cat__Neighborhood_NoRidge      0.085441
107       cat__BldgType_2fmCon      0.084085
85   cat__Neighborhood_Somerst      0.073732
240        cat__Functional_Typ      0.073362
214          cat__Heating_GasW      0.071021
39            cat__MSZoning_RL      0.067302

Top 10 Most Negative Features:
                 Feature_Names  Coefficients
36       cat__MSZoning_C (all)     -0.197898
125      cat__RoofMatl_ClyTile     -0.125996
102       cat__Condition2_PosN     -0.099803
71   cat__Neighborhood_Edwards     -0.090717
235       cat__Functional_Maj2     -0.086983
109        cat__BldgType_Twnhs     -0.084665
74   cat__Neighborhood_MeadowV     -0.081844
75   cat__Neighborhood_Mitchel     -0.076450
73    cat__Neighborhood_IDOTRR     -0

In [308]:
test_data = pd.read_csv("test.csv")
x_test_data = test_data.drop(columns=["Id"])
x_test_data[categorical_columns] = x_test_data[categorical_columns].astype(str).fillna('NA')
x_test_data[numerical_columns] = x_test_data[numerical_columns].astype(float).fillna(0)
x_test_binarized = encoder.transform(x_test_data)
y_test_pred_log = model.predict(x_test_binarized)
y_test_pred = np.exp(y_test_pred_log)
predicted_data = test_data[["Id"]].copy()
predicted_data["SalePrice"] = y_test_pred
predicted_data.to_csv("Linear_regression_p3.csv", index=False)
print("Predicted values saved to 'Linear_regression_p3.csv'")

Predicted values saved to 'Linear_regression_p3.csv'


# Part 4

# Part 4 - 1.1

In [309]:
from sklearn.linear_model import Ridge

encoder = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
encoder.fit(x_train_data)

x_train_binarized = encoder.transform(x_train_data)
x_dev_binarized = encoder.transform(x_dev_data)

alpha_values = [0.01, 0.1, 1, 10, 100]
best_alpha = None
best_rmsle = float('inf')

for alpha in alpha_values:
    ridge = Ridge(alpha=alpha)
    ridge.fit(x_train_binarized, y_train_data)
    y_dev_pred = ridge.predict(x_dev_binarized)
    rmsle = np.sqrt(mean_squared_log_error(np.exp(y_dev_data), np.exp(y_dev_pred)))
    print(f"Alpha: {alpha}, RMSLE: {rmsle}")
    if rmsle < best_rmsle:
        best_rmsle = rmsle
        best_alpha = alpha
        
print(f"Best alpha: {best_alpha}, Best RMSLE: {best_rmsle}")

Alpha: 0.01, RMSLE: 0.15009741232254975
Alpha: 0.1, RMSLE: 0.14475195225357734
Alpha: 1, RMSLE: 0.14064027497926507
Alpha: 10, RMSLE: 0.13945557653847268
Alpha: 100, RMSLE: 0.15429049111662646
Best alpha: 10, Best RMSLE: 0.13945557653847268


In [310]:
best_ridge_model = Ridge(alpha=best_alpha)
best_ridge_model.fit(x_train_binarized, y_train_data)

y_dev_pred = best_ridge_model.predict(x_dev_binarized)

rmsle_ridge = np.sqrt(mean_squared_log_error(np.exp(y_dev_data), np.exp(y_dev_pred)))
print(f"RMSLE on dev: {rmsle_ridge}")


RMSLE on dev: 0.13945557653847268


In [311]:
test_data = pd.read_csv("test.csv")
x_test_data = test_data.drop(columns=["Id"]).astype(str)
x_test_binarized = encoder.transform(x_test_data)

y_test_pred_log = best_ridge_model.predict(x_test_binarized)
y_test_pred = np.exp(y_test_pred_log)

predicted_data = test_data[["Id"]].copy()
predicted_data["SalePrice"] = y_test_pred
predicted_data.to_csv("Linear_regression_p4-1_1.csv", index=False)
print("Predicted values saved to Linear_regression_p4-1_1.csv")

Predicted values saved to Linear_regression_p4-1_1.csv


# Part 4 - 1.2

In [312]:
from sklearn.compose import ColumnTransformer

numerical_columns = [
    "MSSubClass", "LotArea", "OverallQual", "OverallCond", "YearBuilt", 
    "YearRemodAdd", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", 
    "TotalBsmtSF", "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "GrLivArea", 
    "BsmtFullBath", "BsmtHalfBath", "FullBath", "HalfBath", "BedroomAbvGr", 
    "KitchenAbvGr", "TotRmsAbvGrd", "Fireplaces", "GarageCars", 
    "GarageArea", "WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch", 
    "ScreenPorch", "PoolArea", "MiscVal", "MoSold", "YrSold", "LotFrontage", "GarageYrBlt", "MasVnrArea"
]

all_columns = train_data.columns.tolist()
categorical_columns = [col for col in all_columns if col not in ["Id", "SalePrice"] + numerical_columns]

x_train_data[numerical_columns] = x_train_data[numerical_columns].astype(float).fillna(0)
x_dev_data[numerical_columns] = x_dev_data[numerical_columns].astype(float).fillna(0)
x_train_data[categorical_columns] = x_train_data[categorical_columns].astype(str).fillna('NA')
x_dev_data[categorical_columns] = x_dev_data[categorical_columns].astype(str).fillna('NA')

In [313]:
encoder = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_columns), 
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), categorical_columns)
    ]
)
encoder.fit(x_train_data)

ColumnTransformer(transformers=[('num', 'passthrough',
                                 ['MSSubClass', 'LotArea', 'OverallQual',
                                  'OverallCond', 'YearBuilt', 'YearRemodAdd',
                                  'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
                                  'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
                                  'LowQualFinSF', 'GrLivArea', 'BsmtFullBath',
                                  'BsmtHalfBath', 'FullBath', 'HalfBath',
                                  'BedroomAbvGr', 'KitchenAbvGr',
                                  'TotRmsAbvGrd', 'Fireplaces', 'Garage...
                                 ['MSZoning', 'Street', 'Alley', 'LotShape',
                                  'LandContour', 'Utilities', 'LotConfig',
                                  'LandSlope', 'Neighborhood', 'Condition1',
                                  'Condition2', 'BldgType', 'HouseStyle',
                                  'RoofStyle', 'RoofMatl', 'Exterior1st',
                                  'Exterior2nd', 'MasVnrType', 'ExterQual',
                                  'ExterCond', 'Foundation', 'BsmtQual',
                                  'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
                                  'BsmtFinType2', 'Heating', 'HeatingQC',
                                  'CentralAir', 'Electrical', ...])])

In [314]:
x_train_binarized = encoder.transform(x_train_data)
x_dev_binarized = encoder.transform(x_dev_data)

alpha_values = [0.01, 0.1, 1, 10, 100]
best_alpha = None
best_rmsle = float('inf')

for alpha in alpha_values:
    ridge = Ridge(alpha=alpha)
    ridge.fit(x_train_binarized, y_train_data)

    y_dev_pred = ridge.predict(x_dev_binarized)

    rmsle = np.sqrt(mean_squared_log_error(np.exp(y_dev_data), np.exp(y_dev_pred)))
    print(f"Alpha: {alpha}, RMSLE: {rmsle}")

    if rmsle < best_rmsle:
        best_rmsle = rmsle
        best_alpha = alpha

print(f"Best alpha: {best_alpha}, Best RMSLE: {best_rmsle}")

Alpha: 0.01, RMSLE: 0.12445959990875573
Alpha: 0.1, RMSLE: 0.1252547256933347
Alpha: 1, RMSLE: 0.1279748340544622
Alpha: 10, RMSLE: 0.1275838984055704
Alpha: 100, RMSLE: 0.12844640101386404
Best alpha: 0.01, Best RMSLE: 0.12445959990875573


In [315]:
best_ridge_model = Ridge(alpha=best_alpha)
best_ridge_model.fit(x_train_binarized, y_train_data)

y_dev_pred = best_ridge_model.predict(x_dev_binarized)

rmsle_ridge = np.sqrt(mean_squared_log_error(np.exp(y_dev_data), np.exp(y_dev_pred)))
print(f"RMSLE on dev: {rmsle_ridge}")

RMSLE on dev: 0.12445959990875573


In [316]:
test_data = pd.read_csv("test.csv")
x_test_data = test_data.drop(columns=["Id"])
x_test_data[categorical_columns] = x_test_data[categorical_columns].astype(str).fillna('NA')
x_test_data[numerical_columns] = x_test_data[numerical_columns].astype(float).fillna(0)
x_test_binarized = encoder.transform(x_test_data)

y_test_pred_log = best_ridge_model.predict(x_test_binarized)
y_test_pred = np.exp(y_test_pred_log)

predicted_data = test_data[["Id"]].copy()
predicted_data["SalePrice"] = y_test_pred
predicted_data.to_csv("Linear_regression_p4-1_2.csv", index=False)
print("Predicted values saved to Linear_regression_p4-1_2.csv")

Predicted values saved to Linear_regression_p4-1_2.csv


# Part 4 - 2

In [317]:
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures
from sklearn.pipeline import make_pipeline

numerical_columns = [
    "LotArea", "OverallQual", "OverallCond", "YearBuilt", 
    "YearRemodAdd", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", 
    "TotalBsmtSF", "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "GrLivArea", 
    "BsmtFullBath", "BsmtHalfBath", "FullBath", "HalfBath", "BedroomAbvGr", 
    "KitchenAbvGr", "TotRmsAbvGrd", "Fireplaces", "GarageCars", 
    "GarageArea", "WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch", 
    "ScreenPorch", "PoolArea", "MiscVal", "LotFrontage", "GarageYrBlt", "MasVnrArea","MSSubClass", "MoSold", "YrSold"
]

new_x_train = train_data.drop(columns=["Id", "SalePrice"])
new_y_train = np.log(train_data['SalePrice'])

new_x_dev = dev_data.drop(columns=["Id", "SalePrice"])
new_y_dev = np.log(dev_data['SalePrice'])

all_columns = train_data.columns.tolist()
categorical_columns = [col for col in all_columns if col not in ["Id", "SalePrice"] + numerical_columns ]

In [318]:
new_x_train[numerical_columns] = new_x_train[numerical_columns].astype(float)
new_x_dev[numerical_columns] = new_x_dev[numerical_columns].astype(float)
new_x_test = test_data.drop(columns=["Id"])
new_x_test[numerical_columns] = new_x_test[numerical_columns].astype(float)

median_values = new_x_train[numerical_columns].median()
new_x_train[numerical_columns] = new_x_train[numerical_columns].fillna(median_values)
new_x_dev[numerical_columns] = new_x_dev[numerical_columns].fillna(median_values)
new_x_test[numerical_columns] = new_x_test[numerical_columns].fillna(median_values)

new_x_train[categorical_columns] = new_x_train[categorical_columns].astype(str).fillna('NA')
new_x_dev[categorical_columns] = new_x_dev[categorical_columns].astype(str).fillna('NA')
new_x_test[categorical_columns] = new_x_test[categorical_columns].astype(str).fillna('NA')

In [319]:
important_numerical_columns = ["OverallQual", "GrLivArea", "TotalBsmtSF", "GarageArea", "1stFlrSF", "FullBath", "YearBuilt","LotFrontage","MSSubClass"]


numerical_pipeline = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False)
)

categorical_pipeline = make_pipeline(
    OneHotEncoder(sparse_output=False, handle_unknown='ignore')
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline, important_numerical_columns),
        ('cat', categorical_pipeline, categorical_columns)
    ]
)

model_pipeline = make_pipeline(
    preprocessor,
    LinearRegression()
)

model_pipeline.fit(new_x_train, new_y_train)

y_dev_pred = model_pipeline.predict(new_x_dev)

rmsle = np.sqrt(mean_squared_log_error(np.exp(new_y_dev), np.exp(y_dev_pred)))
print(f"RMSLE on dev: {rmsle}")

y_test_pred_log = model_pipeline.predict(new_x_test)
y_test_pred = np.exp(y_test_pred_log)


predicted_data = test_data[["Id"]].copy()
predicted_data["SalePrice"] = y_test_pred
predicted_data.to_csv("Linear_regression_p4-2.CSV", index=False)
print("Predicted values saved to 'Linear_regression_p4-2.csv'")

RMSLE on dev: 0.13477290917288706
Predicted values saved to 'Linear_regression_p4-2.csv'


# Part 4: 4 - Final Code

In [320]:
from sklearn.preprocessing import StandardScaler

numerical_pipeline = make_pipeline(
    StandardScaler(),
    PolynomialFeatures(degree=2, include_bias=False)
)

categorical_pipeline = make_pipeline(
    OneHotEncoder(sparse_output=True, handle_unknown='ignore')
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline, important_numerical_columns),
        ('cat', categorical_pipeline, categorical_columns)
    ]
)

preprocessor.fit(new_x_train)

x_train_binarized = preprocessor.transform(new_x_train)
x_dev_binarized = preprocessor.transform(new_x_dev)

In [321]:
alpha_values = [0.01, 0.1, 1, 5, 10, 13, 50, 100]
best_alpha = None
best_rmsle = float('inf')
        
for alpha in alpha_values:
    ridge = Ridge(alpha=alpha)
    ridge.fit(x_train_binarized, y_train_data)

    y_dev_pred = ridge.predict(x_dev_binarized)

    rmsle = np.sqrt(mean_squared_log_error(np.exp(y_dev_data), np.exp(y_dev_pred)))
    print(f"Alpha: {alpha}, RMSLE: {rmsle}")

    if rmsle < best_rmsle:
        best_rmsle = rmsle
        best_alpha = alpha
    
print(f"Best alpha: {best_alpha}")

Alpha: 0.01, RMSLE: 0.1348926672581257
Alpha: 0.1, RMSLE: 0.1349337335490633
Alpha: 1, RMSLE: 0.13402763813905244
Alpha: 5, RMSLE: 0.13223916081433598
Alpha: 10, RMSLE: 0.13168783354296615
Alpha: 13, RMSLE: 0.1316075728347801
Alpha: 50, RMSLE: 0.13284765919807878
Alpha: 100, RMSLE: 0.1348462103307313
Best alpha: 13


In [322]:
best_ridge_model = Ridge(alpha=best_alpha, solver="auto")
best_ridge_model.fit(x_train_binarized, y_train_data)
y_dev_pred = best_ridge_model.predict(x_dev_binarized)
rmsle = np.sqrt(mean_squared_log_error(np.exp(new_y_dev), np.exp(y_dev_pred)))
print(f"RMSLE on dev: {rmsle}")

RMSLE on dev: 0.1316075728347801


In [323]:
test_data_X_binarized = preprocessor.transform(new_x_test)
y_test_pred_log = best_ridge_model.predict(test_data_X_binarized)
y_test_pred = np.exp(y_test_pred_log)

predicted_data = test_data[["Id"]].copy()
predicted_data["SalePrice"] = y_test_pred
predicted_data.to_csv("Linear_regression_p4-4.csv", index=False)
print("Predicted values saved to 'Linear_regression_p4-4.csv'")

Predicted values saved to 'Linear_regression_p4-4.csv'
